# 04 · Classify individual local files

Send a file's **bytes** straight from disk to the classifier — no storage account
involved. Point `FILE_PATH` at any local PDF/image/Office document to try your own.

The response carries a **confidence** per predicted `docType`, which is the built-in
signal you can threshold for auto-accept vs. human review.

## 1. Setup

In [ ]:
import os, sys
from pathlib import Path

_here = Path.cwd()
for _c in (_here, _here / 'notebooks', *_here.parents):
    if (_c / '_lib.py').exists():
        sys.path.insert(0, str(_c))
        break
import _lib

di = _lib.get_di_client()
classifier_id = _lib.CLASSIFIER_ID

## 2. Classify one local file

Change `FILE_PATH` to classify your own document.

In [ ]:
FILE_PATH = _lib.DATA_DIR / 'FabrikamInvoice_1.pdf'   # <- change to any local file

with open(FILE_PATH, 'rb') as f:
    result = di.begin_classify_document(classifier_id, body=f, split='none').result()

_lib.print_result(result, source=FILE_PATH.name)
_lib.save_result(result, _lib.SAMPLE_OUTPUT_DIR / f'local_{FILE_PATH.stem}.json')

## 3. Classify a batch of local files

In [ ]:
files = [
    _lib.DATA_DIR / 'FabrikamInvoice_1.pdf',
    _lib.DATA_DIR / 'ClaimForm_1.pdf',
    _lib.CLAIMS_DIR / 'claimform_handwritten_1.pdf',
]

for path in files:
    with open(path, 'rb') as f:
        result = di.begin_classify_document(classifier_id, body=f, split='none').result()
    _lib.print_result(result, source=path.name)
    print()

## 4. (Optional) Accuracy + confidence over all samples

Classifies every file in both class folders, compares the prediction to the known
class, and buckets by confidence — the same confidence-to-automation idea used
elsewhere in this repo. Set `RUN_EVAL = True` to run (one API call per file).

> Note: these files were also used for training, so accuracy here is optimistic —
> it demonstrates the confidence signal, not held-out performance.

In [ ]:
RUN_EVAL = False   # set True to score every sample

def bucket(v):
    if v is None:
        return 'none'
    if v > 0.95:
        return 'high'
    if v >= 0.70:
        return 'mid'
    return 'low'

if RUN_EVAL:
    from collections import defaultdict
    stats = defaultdict(lambda: {'count': 0, 'correct': 0})
    total = 0
    correct = 0
    for label, folder in _lib.CLASS_DIRS.items():
        for path in sorted(Path(folder).glob('*.pdf')):
            with open(path, 'rb') as f:
                r = di.begin_classify_document(classifier_id, body=f, split='none').result()
            rows = _lib.summarize(r)
            pred = rows[0]['doc_type'] if rows else None
            conf = rows[0]['confidence'] if rows else None
            ok = (pred == label)
            total += 1
            correct += int(ok)
            b = bucket(conf)
            stats[b]['count'] += 1
            stats[b]['correct'] += int(ok)
            status = 'OK' if ok else 'MISS'
            print(f'  {path.name:32s} true={label:8s} pred={str(pred):8s} conf={conf}  {status}')
    print()
    print(f'accuracy: {correct}/{total} = {(correct / total * 100) if total else 0:.1f}%')
    for b in ('high', 'mid', 'low', 'none'):
        s = stats[b]
        if s['count']:
            print(f'  {b:5s}: {s["count"]:3d} docs, {s["correct"]} correct')
else:
    print('RUN_EVAL is False — set it to True to score all samples.')